# CSV Benchmarks – Backtesting Experiments

## Libraries

In [1]:
from fabric import ThreadingGroup, Connection
from chi import server, ssh
import os
import warnings
import time
import sys
sys.path.append('../scripts')
from utilities import *
warnings.filterwarnings('ignore')

## Configurations

In [2]:
sshUser = "ubuntu"
server_ips = ["64.131.114.155"]
server_local_ips = {}
client_ips = ["64.131.114.143", "64.131.114.145", "64.131.114.146", "64.131.114.147",
              "64.131.114.148", "64.131.114.152", "64.131.114.153", "64.131.114.154"]

storageType = {"HDD": "/HDD", "NVME": "/NVME_RAID-0"}
SERVER_WORKING_DIR = storageType["NVME"]

targetClients = [1, 2, 4, 8, 16, 32, 64]

## Initialize environment

In [3]:
# These groups are for processing in parallel 
serverConnections = [makeConnection(i, user=sshUser) for i in server_ips]
clientConnections = [makeConnection(i, user=sshUser) for i in client_ips]
allConnections = (serverConnections + clientConnections)
serverGroup = ThreadingGroup.from_connections(serverConnections)
clientGroup = ThreadingGroup.from_connections(clientConnections)
allGroup = ThreadingGroup.from_connections(allConnections)

# Get local IP for server
local_ips = serverGroup.run("hostname -I | cut -d ' ' -f 1", hide=True)
for i in local_ips:
    server_local_ips[i.host] = local_ips[i].stdout.strip()

# Update & Upgrade latest packages
allGroup.run("sudo apt update && sudo apt upgrade -y", hide=True)
rebootGroup(allGroup)

Wait for all server to come online after reboot

In [4]:
# Buffer of reboot (no more +1 min shutdown-schedule delay)
time.sleep(240)

for ip in (server_ips + client_ips):
    server.wait_for_tcp(ip, 22, timeout=1200)

## Benchmark server

Prepare benchmark server

In [5]:
serverGroup.put("../scripts/NFS/initNFSServer.sh")
serverGroup.run(f"sudo chmod -R 777 {SERVER_WORKING_DIR}; bash initNFSServer.sh {SERVER_WORKING_DIR}", hide=True);

## Benchmark Client (Backtester)

In [6]:
clientGroup.put("../scripts/backtesterSkipping.py")
clientGroup.put("../scripts/initBacktestClient.sh")
clientGroup.put("../scripts/NFS/initNFSClient.sh")
clientGroup.run("bash initBacktestClient.sh", hide=True)
clientGroup.run(f"bash initNFSClient.sh {server_local_ips[server_ips[0]]}", hide=True)
clientGroup.close()

## Bootstrap Database

In [7]:
serverGroup.put("../scripts/restoreDB.sh")

# Restore DB
serverGroup.run(f"bash restoreDB.sh MYSTIC@IIT CSV BACKTESTER {SERVER_WORKING_DIR}", hide=True)

# Keep checking if job is done running in tmux window
checkTmuxJobIsDone(serverGroup);

## Experiments

**Perform experiments**

In [8]:
runWorkload("BACKTEST", targetClients, server_local_ips[server_ips[0]], 0000, "CSV", client_ips, 100, 10, serverGroup, SERVER_WORKING_DIR, "ubuntu", 1000000, 900, "/NFS_STORAGE/BacktestData.csv", sleepInterval=60)

[Benchmark Config]
  Target Clients     : 1
  Total Machines     : 8
  Machines Used      : 1
  Threads per Machine: 1
  Total Clients Actual : 1
[Benchmark: 1] - Schedule workload on 1776144476
[Benchmark: 1] [Client: 64.131.114.143] - Time now: 1776144363 | Delta: 113
[Benchmark: 1] [Client: 64.131.114.143] - Downloaded 1 result files
[Benchmark Config]
  Target Clients     : 2
  Total Machines     : 8
  Machines Used      : 2
  Threads per Machine: 1
  Total Clients Actual : 2
[Benchmark: 2] - Schedule workload on 1776145628
[Benchmark: 2] [Client: 64.131.114.143] - Time now: 1776145517 | Delta: 111
[Benchmark: 2] [Client: 64.131.114.145] - Time now: 1776145524 | Delta: 104
[Benchmark: 2] [Client: 64.131.114.143] - Downloaded 1 result files
[Benchmark: 2] [Client: 64.131.114.145] - Downloaded 1 result files
[Benchmark Config]
  Target Clients     : 4
  Total Machines     : 8
  Machines Used      : 4
  Threads per Machine: 1
  Total Clients Actual : 4
[Benchmark: 4] - Schedule worklo